### 0. Setup

In [2]:
# ==============================================================================
# CELLULE 1 : SETUP, PARAMÈTRES GLOBAUX ET CHARGEMENT DES DONNÉES GDELT
# ==============================================================================
import csv
from datetime import datetime
import io
from pathlib import Path
import warnings
from group_lasso import GroupLasso
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import requests
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

# --- PARAMÈTRES GLOBAUX ---
START_DATE = datetime(2013, 1, 1)
END_DATE = datetime(2026, 7, 1)
MIN_INDEX_DATE = '2015-03-01'

# --- CHARGEMENT GDELT (FRANCE / US / UK) ---
dir_geo = Path('./data/indicators_geo_monthly')
dir_geo_uk = Path('./data/indicators_geo_monthly_UK')

# 1. Chargement France / US
if dir_geo.exists():
  files_geo = list(dir_geo.glob('*.parquet'))
  df_geo_main = pd.concat(
      [pd.read_parquet(f) for f in files_geo], ignore_index=True
  )
else:
  raise FileNotFoundError('⚠ Dossier ./data/indicators_geo_monthly introuvable.')

# 2. Chargement UK
if dir_geo_uk.exists():
  files_uk = list(dir_geo_uk.glob('*.parquet'))
  if len(files_uk) > 0:
    df_geo_uk = pd.concat(
        [pd.read_parquet(f) for f in files_uk], ignore_index=True
    )
    df_geo_uk['region_key'] = 'UK'
  else:
    df_geo_uk = pd.DataFrame()
else:
  df_geo_uk = pd.DataFrame()

# 3. Fusion finale
df_geo = pd.concat([df_geo_main, df_geo_uk], ignore_index=True)

if 'region_key' in df_geo.columns:
  df_geo = (
      df_geo.groupby(['period', 'region_key']).max().reset_index().copy()
  )
  df_geo['period'] = pd.to_datetime(df_geo['period'])
  print(f'✓ Base RÉGIONALE GDELT chargée.')
  print(
      f"✓ Régions trouvées en mémoire : {df_geo['region_key'].unique().tolist()}"
  )
  print(f'✓ Lignes totales : {len(df_geo)}')
else:
  print("⚠ ERREUR CRITIQUE : La colonne 'region_key' est absente.")

✓ Base RÉGIONALE GDELT chargée.
✓ Régions trouvées en mémoire : ['France', 'UK', 'US']
✓ Lignes totales : 411


### Functions

In [20]:
# ==============================================================================
# CELLULE 2 : LIBRAIRIE DES FONCTIONS ÉCONOMÉTRIQUES ET DE PRÉPARATION
# ==============================================================================

def get_filtered_gdelt_cols(df):
    """ÉTAPE 0 : Ne conserve que les sous-variables des 8 secteurs cibles."""
    allowed_prefixes = [
        'att_weight_agriculture_', 'att_weight_commodities_',
        'att_weight_energy_', 'att_weight_finance_',
        'att_weight_industry_', 'att_weight_real_estate_',
        'att_weight_tech_', 'att_weight_transport_', 
        'att_weight_growth_', 'att_weight_conflict_', 'att_weight_politics_',
        'att_weight_health_', 'att_weight_consumer_goods_'
    ]
    return [col for col in df.columns if any(str(col).startswith(prefix) for prefix in allowed_prefixes)]

def strict_stationarize(df):
    """ÉTAPE 1 : Test ADF -> Différence -> Winsorisation si nécessaire."""
    df_stat = df.copy()
    diff_count = 0
    for col in df_stat.columns:
        valid_data = df_stat[col].dropna()
        if len(valid_data) > 10:
            if adfuller(valid_data, autolag='AIC')[1] >= 0.05:
                df_stat[col] = df_stat[col].diff()
                diff_count += 1
                valid_data_diff = df_stat[col].dropna()
                if len(valid_data_diff) > 10 and adfuller(valid_data_diff, autolag='AIC')[1] >= 0.05:
                    p95 = valid_data_diff.quantile(0.95)
                    p05 = valid_data_diff.quantile(0.05)
                    df_stat[col] = df_stat[col].clip(lower=p05, upper=p95)
    return df_stat.dropna(), diff_count

def fetch_and_prep_macro_dynamic(region_config, start_date, end_date, region=None):
    """Télécharge la macro et génère 12 lags (pour fixer la taille de l'échantillon)."""
    macro_config = region_config['macro_vars']
    fred_tickers = {info['ticker']: var_name for var_name, info in macro_config.items() 
                    if not (region == 'France' and var_name == 'money') and 
                    not (region == 'UK' and info['ticker'] in ['IUMABEDR', 'LPMBD93', 'XUDLUSS'])}
    
    df = web.DataReader(list(fred_tickers.keys()), 'fred', start_date, end_date).rename(columns=fred_tickers).resample('MS').mean() if fred_tickers else pd.DataFrame()

    if region == 'France' and 'money' in macro_config:
        try:
            df_ecb = pd.read_csv('https://data-api.ecb.europa.eu/service/data/BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E?format=csvdata')
            df_ecb['period'] = pd.to_datetime(df_ecb['TIME_PERIOD'])
            df_ecb = df_ecb.set_index('period').rename(columns={'OBS_VALUE': 'money'})[['money']].resample('MS').mean()
            df = df.join(df_ecb, how='left') if not df.empty else df_ecb
        except Exception as e: print(f'⚠ Erreur BCE: {e}')

    if region == 'UK':
        df_local_all = pd.DataFrame()
        for var_name, info in macro_config.items():
            if info['ticker'] in ['IUMABEDR', 'LPMBD93', 'XUDLUSS']:
                ticker = info['ticker']
                file_path = f'data/BoE/{ticker}.csv'
                if Path(file_path).exists():
                    df_temp = pd.read_csv(file_path, encoding='ISO-8859-1')
                    df_temp['period'] = pd.to_datetime(df_temp.iloc[:, 0], errors='coerce')
                    df_temp = df_temp.dropna(subset=['period']).set_index('period')
                    val_col = next((c for c in df_temp.columns if ticker in c), None)
                    if val_col:
                        if df_temp[val_col].dtype == object:
                            df_temp[val_col] = df_temp[val_col].astype(str).str.replace(',', '', regex=False).str.strip()
                        series = pd.to_numeric(df_temp[val_col], errors='coerce').rename(var_name).resample('MS').mean()
                        df_local_all = series.to_frame() if df_local_all.empty else df_local_all.join(series, how='outer')
        if not df_local_all.empty: df = df.join(df_local_all, how='outer') if not df.empty else df_local_all

    final_cols = []
    for var_name, info in macro_config.items():
        if var_name in df.columns:
            if info['transform'] == 'pct_change': df[var_name] = df[var_name].pct_change() * 100
            elif info['transform'] == 'diff': df[var_name] = df[var_name].diff()
            final_cols.append(var_name)

    for lag in range(1, 13):
        lag_name = f'inflation_lag{lag}'
        df[lag_name] = df['inflation'].shift(lag)
        final_cols.append(lag_name)

    return df[(df.index >= start_date) & (df.index <= end_date)][final_cols].dropna()

def run_cv_group_lasso(X, y, groups):
    """MODÈLE 1 : Group Lasso (Yuan & Lin, 2006) - Alpha forcé à 0 (Pénalité de groupe pure)."""
    tscv = TimeSeriesSplit(n_splits=5, gap=3) 
    lambdas = np.logspace(-4, -1, 50)
    best_mse, best_lbd = np.inf, None

    for lbd in lambdas:
        gl = GroupLasso(groups=groups, l1_reg=0.0, group_reg=lbd, fit_intercept=False, n_iter=1000, supress_warning=True)
        mse_scores = []
        for train_idx, test_idx in tscv.split(X):
            gl.fit(X.iloc[train_idx].values, y.iloc[train_idx].values.reshape(-1, 1))
            mse_scores.append(np.mean((y.iloc[test_idx].values - gl.predict(X.iloc[test_idx].values).flatten()) ** 2))
        
        if np.mean(mse_scores) < best_mse:
            best_mse, best_lbd = np.mean(mse_scores), lbd

    final_gl = GroupLasso(groups=groups, l1_reg=0.0, group_reg=best_lbd, fit_intercept=False, n_iter=5000, supress_warning=True)
    final_gl.fit(X.values, y.values.reshape(-1, 1))
    return final_gl.coef_.flatten(), best_lbd

def run_cv_sparse_group_lasso(X, y, groups):
    """MODÈLE 2 : Sparse Group Lasso (Simon et al., 2013) - Tuning de Lambda ET Alpha."""
    tscv = TimeSeriesSplit(n_splits=5, gap=3) 
    lambdas = np.logspace(-4, -1, 50)
    alphas = [0.05, 0.25, 0.50, 0.75, 0.95] # Grille Data-Driven
    
    best_mse, best_lbd, best_alpha = np.inf, None, None

    for alpha in alphas:
        for lbd in lambdas:
            sgl = GroupLasso(groups=groups, l1_reg=lbd * alpha, group_reg=lbd * (1 - alpha), fit_intercept=False, n_iter=1000, supress_warning=True)
            mse_scores = []
            for train_idx, test_idx in tscv.split(X):
                sgl.fit(X.iloc[train_idx].values, y.iloc[train_idx].values.reshape(-1, 1))
                mse_scores.append(np.mean((y.iloc[test_idx].values - sgl.predict(X.iloc[test_idx].values).flatten()) ** 2))
            
            if np.mean(mse_scores) < best_mse:
                best_mse, best_lbd, best_alpha = np.mean(mse_scores), lbd, alpha

    final_sgl = GroupLasso(groups=groups, l1_reg=best_lbd * best_alpha, group_reg=best_lbd * (1 - best_alpha), fit_intercept=False, n_iter=5000, supress_warning=True)
    final_sgl.fit(X.values, y.values.reshape(-1, 1))
    return final_sgl.coef_.flatten(), best_lbd, best_alpha

def run_post_lasso_ols(X, y, selected_features):
    """Inférence OLS Post-Selection et diagnostic de Ljung-Box."""
    X_selected_with_const = sm.add_constant(X[selected_features])
    results = sm.OLS(y.values, X_selected_with_const.values).fit()

    post_lasso_df = pd.DataFrame({
        'Variable': ['Intercept'] + selected_features,
        'Coefficient': results.params,
        'P-value': results.pvalues,
        'CI_Lower': results.conf_int()[:, 0],
        'CI_Upper': results.conf_int()[:, 1],
    })
    
    lb_pval = acorr_ljungbox(results.resid, lags=[12], return_df=True)['lb_pvalue'].values[0]
    return post_lasso_df, results.rsquared, results.rsquared_adj, lb_pval

### 1. Inflation 

In [28]:
# ==============================================================================
# CELLULE 3 : CONFIGURATION DES MODÈLES RÉGIONAUX
# ==============================================================================
CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            'inflation': {'ticker': 'CP0000FRM086NEST', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': True,
    },
    'US': {
        'macro_vars': {
            'inflation': {'ticker': 'CPIAUCSL', 'transform': 'pct_change'},
            'rate': {'ticker': 'FEDFUNDS', 'transform': 'diff'},
            'unemp': {'ticker': 'UNRATE', 'transform': 'diff'},
            'indpro': {'ticker': 'INDPRO', 'transform': 'pct_change'},
            'money': {'ticker': 'M2SL', 'transform': 'pct_change'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'MICH', 'transform': 'diff'},
            'oil': {'ticker': 'WTISPLC', 'transform': 'pct_change'},
        },
        'deseasonalize_y': False,
    },
    'UK': {
        'macro_vars': {
            'inflation': {'ticker': 'GBRCPIALLMINMEI', 'transform': 'pct_change'},
            'rate': {'ticker': 'IUMABEDR', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTGBM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'GBRPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'LPMBD93', 'transform': 'pct_change'},
            'fx': {'ticker': 'XUDLUSS', 'transform': 'pct_change'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': True,
    },
}

In [30]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET AVEC COMPARAISON GL vs SGL
# ==============================================================================
final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f'\n' + '=' * 80)
    print(f' PIPELINE ACADÉMIQUE : {region.upper()} (RECHERCHE DYNAMIQUE DE LAGS)')
    print(f'================================================================================')

    # 1. Préparation des données (Tronquées pour garantir un 'n' constant)
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]

    # --- NOUVEAU : SAUVEGARDE VINTAGE ---
    vintage_path = Path(f'./data/economic_data/macro_vintage_inflation_{region}_{datetime.now().strftime("%Y%m%d")}.csv')
    df_macro.to_csv(vintage_path)
    print(f'💾 Données Macro sauvegardées : {vintage_path}')
    # ------------------------------------

    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    if 'att_weight_finance_international_orgs' in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=['att_weight_finance_international_orgs'])

    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']

    y_corr = (y_raw - seasonal_decompose(y_raw, model='additive', period=12).seasonal).dropna() if config['deseasonalize_y'] else y_raw.dropna()
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')

    y_centered = y_corr - y_corr.mean()
    X_scaled_full = pd.DataFrame(StandardScaler().fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)

    macro_base_cols = [c for c in X_scaled_full.columns if c in df_macro.columns and not c.startswith('inflation_lag') and c != 'inflation']
    gdelt_cols = [c for c in X_scaled_full.columns if c in cols_gdelt]

    # --- RECHERCHE DYNAMIQUE DU NOMBRE DE LAGS ---
    max_lags_to_test = 12
    best_iteration_gl = None
    best_iteration_sgl = None

    for current_lag in range(1, max_lags_to_test + 1):
        active_lags = [f'inflation_lag{i}' for i in range(1, current_lag + 1)]
        current_cols = macro_base_cols + active_lags + gdelt_cols
        X_current = X_scaled_full[current_cols]
        
        # Définition des groupes
        groups, group_map, current_group_id = [], {}, 2
        for var in current_cols:
            if var in macro_base_cols: groups.append(0) 
            elif var in active_lags: groups.append(1) 
            else:
                sector = var.split('_')[2]
                if sector not in group_map:
                    group_map[sector], current_group_id = current_group_id, current_group_id + 1
                groups.append(group_map[sector])
        groups_array = np.array(groups)

        # ---------------------------------------------------------
        # MODÈLE 1 : GROUP LASSO (Yuan & Lin)
        # ---------------------------------------------------------
        coefs_gl, lbd_gl = run_cv_group_lasso(X_current, y_centered, groups_array)
        selected_vars_gl = [current_cols[i] for i in range(len(current_cols)) if coefs_gl[i] != 0]
        final_vars_gl = list(set(macro_base_cols + active_lags + selected_vars_gl))
        final_vars_gl_sorted = [v for v in macro_base_cols if v in final_vars_gl] + [v for v in active_lags if v in final_vars_gl] + sorted([v for v in gdelt_cols if v in final_vars_gl])
        df_gl, r2_gl, r2_adj_gl, lb_pval_gl = run_post_lasso_ols(X_current, y_centered, final_vars_gl_sorted)
        best_iteration_gl = (current_lag, final_vars_gl_sorted, df_gl, r2_gl, r2_adj_gl, lb_pval_gl, lbd_gl)

        # ---------------------------------------------------------
        # MODÈLE 2 : SPARSE GROUP LASSO (Simon et al.)
        # ---------------------------------------------------------
        coefs_sgl, lbd_sgl, alpha_sgl = run_cv_sparse_group_lasso(X_current, y_centered, groups_array)
        selected_vars_sgl = [current_cols[i] for i in range(len(current_cols)) if coefs_sgl[i] != 0]
        final_vars_sgl = list(set(macro_base_cols + active_lags + selected_vars_sgl))
        final_vars_sgl_sorted = [v for v in macro_base_cols if v in final_vars_sgl] + [v for v in active_lags if v in final_vars_sgl] + sorted([v for v in gdelt_cols if v in final_vars_sgl])
        df_sgl, r2_sgl, r2_adj_sgl, lb_pval_sgl = run_post_lasso_ols(X_current, y_centered, final_vars_sgl_sorted)
        best_iteration_sgl = (current_lag, final_vars_sgl_sorted, df_sgl, r2_sgl, r2_adj_sgl, lb_pval_sgl, lbd_sgl, alpha_sgl)

        # Règle d'arrêt : On s'arrête dès que le SGL capture toute la dynamique temporelle (Bruit blanc validé)
        if lb_pval_sgl >= 0.05:
            print(f'  ✓ Bruit blanc validé au Lag {current_lag} (SGL Ljung-Box P-value : {lb_pval_sgl:.4f})')
            break
        else:
            print(f'  ↻ Autocorrélation résiduelle au Lag {current_lag} (P-value : {lb_pval_sgl:.4f}). Ajout du lag {current_lag+1}...')

    # --- AFFICHAGE COMPARATIF DES RÉSULTATS OPTIMAUX ---
    print(f'\n' + '=' * 80)
    print(f' RÉSULTATS COMPARATIFS FINAUX : {region.upper()} (STABILISÉ À {best_iteration_sgl[0]} LAG(S))')
    print(f'=' * 80)

    # Affichage Group Lasso
    lag_gl, vars_gl, df_gl, r2_gl, r2_adj_gl, pval_gl, lbd_gl = best_iteration_gl
    gdelt_gl = [v for v in vars_gl if v in gdelt_cols]
    
    print(f'\n▶ MODÈLE 1 : GROUP LASSO (Yuan & Lin, 2006) - Sélection de secteurs entiers')
    print(f'  Tuning Data-Driven : Lambda = {lbd_gl:.4f} | Alpha forcé = 0.0')
    print(f'  Variables totales (OLS) : {len(vars_gl)} (dont {len(gdelt_gl)} GDELT)')
    print(f'  R-squared (Global/Adj) : {r2_gl:.4f} / {r2_adj_gl:.4f}')
    print(f'  Ljung-Box P-value : {pval_gl:.4f}')
    print("-" * 60)
    print(df_gl.to_string(index=False))

    # Affichage Sparse Group Lasso
    lag_sgl, vars_sgl, df_sgl, r2_sgl, r2_adj_sgl, pval_sgl, lbd_sgl, alpha_sgl = best_iteration_sgl
    gdelt_sgl = [v for v in vars_sgl if v in gdelt_cols]
    
    print(f'\n\n▶ MODÈLE 2 : SPARSE GROUP LASSO (Simon et al., 2013) - Sélection intra-secteurs')
    print(f'  Tuning Data-Driven : Lambda = {lbd_sgl:.4f} | Alpha = {alpha_sgl:.2f}')
    print(f'  Variables totales (OLS) : {len(vars_sgl)} (dont {len(gdelt_sgl)} GDELT)')
    print(f'  R-squared (Global/Adj) : {r2_sgl:.4f} / {r2_adj_sgl:.4f}')
    print(f'  Ljung-Box P-value : {pval_sgl:.4f}')
    print("-" * 60)
    print(df_sgl.to_string(index=False))

    final_results[region] = {
        'GL_Model': {'Vars': vars_gl, 'OLS': df_gl, 'R2_adj': r2_adj_gl},
        'SGL_Model': {'Vars': vars_sgl, 'OLS': df_sgl, 'R2_adj': r2_adj_sgl, 'Optimal_Alpha': alpha_sgl}
    }

    # ==========================================================================
    # EXPORT LATEX AUTOMATIQUE DES TABLEAUX DE RÉSULTATS
    # ==========================================================================
    
    import os
    # Création d'un dossier pour ranger vos tableaux propres
    output_dir = "./latex_tables"
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Export du tableau Group Lasso (GL)
    filename_gl = f"{output_dir}/OLS_GL_{region}.tex"
    # On arrondit à 4 décimales et on supprime l'index (0, 1, 2...) de Pandas
    latex_gl = df_gl.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_gl, 'w') as f:
        f.write(latex_gl)
        
    # 2. Export du tableau Sparse Group Lasso (SGL)
    filename_sgl = f"{output_dir}/OLS_SGL_{region}.tex"
    latex_sgl = df_sgl.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_sgl, 'w') as f:
        f.write(latex_sgl)
        
    # 3. BONUS : Export d'un tableau "Strict" (Uniquement les variables significatives p < 0.10)
    # Très utile pour le corps de votre mémoire (les tableaux complets iront en annexe)
    df_sgl_strict = df_sgl[df_sgl['P-value'] < 0.10]
    filename_sgl_strict = f"{output_dir}/OLS_SGL_{region}_SIGNIFICANT.tex"
    latex_sgl_strict = df_sgl_strict.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_sgl_strict, 'w') as f:
        f.write(latex_sgl_strict)
        
    print(f"  📝 Tableaux LaTeX exportés avec succès dans le dossier '{output_dir}'.")


 PIPELINE ACADÉMIQUE : FRANCE (RECHERCHE DYNAMIQUE DE LAGS)
💾 Données Macro sauvegardées : data/economic_data/macro_vintage_inflation_France_20260826.csv
  ✓ Bruit blanc validé au Lag 1 (SGL Ljung-Box P-value : 0.0808)

 RÉSULTATS COMPARATIFS FINAUX : FRANCE (STABILISÉ À 1 LAG(S))

▶ MODÈLE 1 : GROUP LASSO (Yuan & Lin, 2006) - Sélection de secteurs entiers
  Tuning Data-Driven : Lambda = 0.0079 | Alpha forcé = 0.0
  Variables totales (OLS) : 72 (dont 64 GDELT)
  R-squared (Global/Adj) : 0.7949 / 0.4433
  Ljung-Box P-value : 0.0835
------------------------------------------------------------
                                   Variable   Coefficient  P-value  CI_Lower  CI_Upper
                                  Intercept -2.775558e-16 1.000000 -0.062526  0.062526
                                       rate -1.140080e-02 0.840794 -0.125229  0.102428
                                      unemp  6.592548e-02 0.158550 -0.026755  0.158606
                                     indpro -6.515425

### 2. Core inflation 

In [31]:
# ==============================================================================
# CELLULE 3 : CONFIGURATION DES MODÈLES RÉGIONAUX (CORE INFLATION)
# ==============================================================================
CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            # Mise à jour: Core CPI France (Non-Food, Non-Energy)
            'inflation': {'ticker': 'CPGRLE01FRM659N', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': True,
    },
    'US': {
        'macro_vars': {
            # Mise à jour: Core CPI US (All Items Less Food & Energy)
            'inflation': {'ticker': 'CPILFESL', 'transform': 'pct_change'},
            'rate': {'ticker': 'FEDFUNDS', 'transform': 'diff'},
            'unemp': {'ticker': 'UNRATE', 'transform': 'diff'},
            'indpro': {'ticker': 'INDPRO', 'transform': 'pct_change'},
            'money': {'ticker': 'M2SL', 'transform': 'pct_change'},
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'MICH', 'transform': 'diff'},
            'oil': {'ticker': 'WTISPLC', 'transform': 'pct_change'},
        },
        'deseasonalize_y': False,
    },
    'UK': {
        'macro_vars': {
            'inflation': {'ticker': 'CPGRLE01GBM659N', 'transform': 'pct_change'},
            'rate': {'ticker': 'IUMABEDR', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTGBM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'GBRPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'LPMBD93', 'transform': 'pct_change'},
            'fx': {'ticker': 'XUDLUSS', 'transform': 'pct_change'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'},
        },
        'deseasonalize_y': False,
    },
}

In [32]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET AVEC COMPARAISON GL vs SGL
# ==============================================================================
final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f'\n' + '=' * 80)
    print(f' PIPELINE ACADÉMIQUE : {region.upper()} (RECHERCHE DYNAMIQUE DE LAGS)')
    print(f'================================================================================')

    # 1. Préparation des données (Tronquées pour garantir un 'n' constant)
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]

    # --- NOUVEAU : SAUVEGARDE VINTAGE ---
    vintage_path = Path(f'./data/economic_data/macro_vintage_coreinflation_{region}_{datetime.now().strftime("%Y%m%d")}.csv')
    df_macro.to_csv(vintage_path)
    print(f'💾 Données Macro sauvegardées : {vintage_path}')
    # ------------------------------------

    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    if 'att_weight_finance_international_orgs' in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=['att_weight_finance_international_orgs'])

    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']

    y_corr = (y_raw - seasonal_decompose(y_raw, model='additive', period=12).seasonal).dropna() if config['deseasonalize_y'] else y_raw.dropna()
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')

    y_centered = y_corr - y_corr.mean()
    X_scaled_full = pd.DataFrame(StandardScaler().fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)

    macro_base_cols = [c for c in X_scaled_full.columns if c in df_macro.columns and not c.startswith('inflation_lag') and c != 'inflation']
    gdelt_cols = [c for c in X_scaled_full.columns if c in cols_gdelt]

    # --- RECHERCHE DYNAMIQUE DU NOMBRE DE LAGS ---
    max_lags_to_test = 12
    best_iteration_gl = None
    best_iteration_sgl = None

    for current_lag in range(1, max_lags_to_test + 1):
        active_lags = [f'inflation_lag{i}' for i in range(1, current_lag + 1)]
        current_cols = macro_base_cols + active_lags + gdelt_cols
        X_current = X_scaled_full[current_cols]
        
        # Définition des groupes
        groups, group_map, current_group_id = [], {}, 2
        for var in current_cols:
            if var in macro_base_cols: groups.append(0) 
            elif var in active_lags: groups.append(1) 
            else:
                sector = var.split('_')[2]
                if sector not in group_map:
                    group_map[sector], current_group_id = current_group_id, current_group_id + 1
                groups.append(group_map[sector])
        groups_array = np.array(groups)

        # ---------------------------------------------------------
        # MODÈLE 1 : GROUP LASSO (Yuan & Lin)
        # ---------------------------------------------------------
        coefs_gl, lbd_gl = run_cv_group_lasso(X_current, y_centered, groups_array)
        selected_vars_gl = [current_cols[i] for i in range(len(current_cols)) if coefs_gl[i] != 0]
        final_vars_gl = list(set(macro_base_cols + active_lags + selected_vars_gl))
        final_vars_gl_sorted = [v for v in macro_base_cols if v in final_vars_gl] + [v for v in active_lags if v in final_vars_gl] + sorted([v for v in gdelt_cols if v in final_vars_gl])
        df_gl, r2_gl, r2_adj_gl, lb_pval_gl = run_post_lasso_ols(X_current, y_centered, final_vars_gl_sorted)
        best_iteration_gl = (current_lag, final_vars_gl_sorted, df_gl, r2_gl, r2_adj_gl, lb_pval_gl, lbd_gl)

        # ---------------------------------------------------------
        # MODÈLE 2 : SPARSE GROUP LASSO (Simon et al.)
        # ---------------------------------------------------------
        coefs_sgl, lbd_sgl, alpha_sgl = run_cv_sparse_group_lasso(X_current, y_centered, groups_array)
        selected_vars_sgl = [current_cols[i] for i in range(len(current_cols)) if coefs_sgl[i] != 0]
        final_vars_sgl = list(set(macro_base_cols + active_lags + selected_vars_sgl))
        final_vars_sgl_sorted = [v for v in macro_base_cols if v in final_vars_sgl] + [v for v in active_lags if v in final_vars_sgl] + sorted([v for v in gdelt_cols if v in final_vars_sgl])
        df_sgl, r2_sgl, r2_adj_sgl, lb_pval_sgl = run_post_lasso_ols(X_current, y_centered, final_vars_sgl_sorted)
        best_iteration_sgl = (current_lag, final_vars_sgl_sorted, df_sgl, r2_sgl, r2_adj_sgl, lb_pval_sgl, lbd_sgl, alpha_sgl)

        # Règle d'arrêt : On s'arrête dès que le SGL capture toute la dynamique temporelle (Bruit blanc validé)
        if lb_pval_sgl >= 0.05:
            print(f'  ✓ Bruit blanc validé au Lag {current_lag} (SGL Ljung-Box P-value : {lb_pval_sgl:.4f})')
            break
        else:
            print(f'  ↻ Autocorrélation résiduelle au Lag {current_lag} (P-value : {lb_pval_sgl:.4f}). Ajout du lag {current_lag+1}...')

    # --- AFFICHAGE COMPARATIF DES RÉSULTATS OPTIMAUX ---
    print(f'\n' + '=' * 80)
    print(f' RÉSULTATS COMPARATIFS FINAUX : {region.upper()} (STABILISÉ À {best_iteration_sgl[0]} LAG(S))')
    print(f'=' * 80)

    # Affichage Group Lasso
    lag_gl, vars_gl, df_gl, r2_gl, r2_adj_gl, pval_gl, lbd_gl = best_iteration_gl
    gdelt_gl = [v for v in vars_gl if v in gdelt_cols]
    
    print(f'\n▶ MODÈLE 1 : GROUP LASSO (Yuan & Lin, 2006) - Sélection de secteurs entiers')
    print(f'  Tuning Data-Driven : Lambda = {lbd_gl:.4f} | Alpha forcé = 0.0')
    print(f'  Variables totales (OLS) : {len(vars_gl)} (dont {len(gdelt_gl)} GDELT)')
    print(f'  R-squared (Global/Adj) : {r2_gl:.4f} / {r2_adj_gl:.4f}')
    print(f'  Ljung-Box P-value : {pval_gl:.4f}')
    print("-" * 60)
    print(df_gl.to_string(index=False))

    # Affichage Sparse Group Lasso
    lag_sgl, vars_sgl, df_sgl, r2_sgl, r2_adj_sgl, pval_sgl, lbd_sgl, alpha_sgl = best_iteration_sgl
    gdelt_sgl = [v for v in vars_sgl if v in gdelt_cols]
    
    print(f'\n\n▶ MODÈLE 2 : SPARSE GROUP LASSO (Simon et al., 2013) - Sélection intra-secteurs')
    print(f'  Tuning Data-Driven : Lambda = {lbd_sgl:.4f} | Alpha = {alpha_sgl:.2f}')
    print(f'  Variables totales (OLS) : {len(vars_sgl)} (dont {len(gdelt_sgl)} GDELT)')
    print(f'  R-squared (Global/Adj) : {r2_sgl:.4f} / {r2_adj_sgl:.4f}')
    print(f'  Ljung-Box P-value : {pval_sgl:.4f}')
    print("-" * 60)
    print(df_sgl.to_string(index=False))

    final_results[region] = {
        'GL_Model': {'Vars': vars_gl, 'OLS': df_gl, 'R2_adj': r2_adj_gl},
        'SGL_Model': {'Vars': vars_sgl, 'OLS': df_sgl, 'R2_adj': r2_adj_sgl, 'Optimal_Alpha': alpha_sgl}
    }

    # ==========================================================================
    # EXPORT LATEX AUTOMATIQUE DES TABLEAUX DE RÉSULTATS
    # ==========================================================================
    
    import os
    # Création d'un dossier pour ranger vos tableaux propres
    output_dir = "./latex_tables"
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Export du tableau Group Lasso (GL)
    filename_gl = f"{output_dir}/OLS_GL_core_inflation{region}.tex"
    # On arrondit à 4 décimales et on supprime l'index (0, 1, 2...) de Pandas
    latex_gl = df_gl.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_gl, 'w') as f:
        f.write(latex_gl)
        
    # 2. Export du tableau Sparse Group Lasso (SGL)
    filename_sgl = f"{output_dir}/OLS_SGL_core_inflation{region}.tex"
    latex_sgl = df_sgl.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_sgl, 'w') as f:
        f.write(latex_sgl)
        
    # 3. BONUS : Export d'un tableau "Strict" (Uniquement les variables significatives p < 0.10)
    # Très utile pour le corps de votre mémoire (les tableaux complets iront en annexe)
    df_sgl_strict = df_sgl[df_sgl['P-value'] < 0.10]
    filename_sgl_strict = f"{output_dir}/OLS_SGL_core_inflation{region}_SIGNIFICANT.tex"
    latex_sgl_strict = df_sgl_strict.to_latex(index=False, float_format="%.4f", escape=True)
    with open(filename_sgl_strict, 'w') as f:
        f.write(latex_sgl_strict)
        
    print(f"  📝 Tableaux LaTeX exportés avec succès dans le dossier '{output_dir}'.")


 PIPELINE ACADÉMIQUE : FRANCE (RECHERCHE DYNAMIQUE DE LAGS)
💾 Données Macro sauvegardées : data/economic_data/macro_vintage_coreinflation_France_20260826.csv
  ✓ Bruit blanc validé au Lag 1 (SGL Ljung-Box P-value : 0.1400)

 RÉSULTATS COMPARATIFS FINAUX : FRANCE (STABILISÉ À 1 LAG(S))

▶ MODÈLE 1 : GROUP LASSO (Yuan & Lin, 2006) - Sélection de secteurs entiers
  Tuning Data-Driven : Lambda = 0.1000 | Alpha forcé = 0.0
  Variables totales (OLS) : 76 (dont 68 GDELT)
  R-squared (Global/Adj) : 0.8497 / 0.3928
  Ljung-Box P-value : 0.1400
------------------------------------------------------------
                                   Variable   Coefficient  P-value    CI_Lower   CI_Upper
                                  Intercept  8.076873e-14 1.000000  -10.694374  10.694374
                                       rate -5.242720e+00 0.689885  -31.991004  21.505564
                                      unemp -1.083282e+01 0.239389  -29.343890   7.678252
                                     